# Prerrequisitos

In [4]:
using LinearAlgebra

# Problema de ortogonalización

Considermos el problema de ortogonaliar las columnas de una matriz dada $A$. Los métodos mas usados son
1. Gram-Schmidt clásico (CGS)
2. Gram-Schmidt modificado (MGS)
3. Householder (H) (usando reflexiones). Desarrollado en 1958 por Householder en el Oak Ridge National Laboratory
4. Givens (G) (usando rotaciones). Desarrollado en 1950 por Givens en el Argone National Laboratory.

Además se usa 
5. Reortogonalización 

En aplicaciones practicas CGS es poco usado. MGS es mas estable que CGS. H es el mejor método en términos de estabilidad. G tiene mas calculos pero puede ser usado para matrices esparsas (ralas). En ocaciones debido a errores numéricos o perturbaciones es conveniente reortogonalizar, por ejemplo se puede combinar MGS con reortogonalización.


La ortogonalización de vectores tiene muchas aplicaciones. Por ejemplo en la solución estable de sistemas lineales. Se puede por ejemplo usar ortogonalización para calcular de forma estable la factorización $LU$. Existen métodos iterativos para calculo de vectores propios que son basados en ortogonalización. También es usado en el cálculo de la descomposición SVD. 

Sea $Q\in \mathbb{R}^{m\times p}$, decimos que $Q$ es ortogonal si $Q^TQ=I_{p\times p}$. Si $Q\in \mathbb{C}^{m\times p}$, decimos que $Q$ es unitaria si $Q^*Q=I$. 

Note que si $q_j$, $j=1,2,\dots,p$, denotan las columad de $Q$, entonces $q_j\in \mathbb{R}^m$ y $q_i^Tq_j=\delta_{ij}$. 

# Factorización A=QR

Dada $A\in\mathbb{R}^{m\times n}$ con $A=[a_1,a_2,\dots,a_n]$ donde $a_i, i=1,2,\dots,n$ denotan las columnasd de $A$, queremos escribir 
$$ A= QR$$ donde  $Q\in\mathbb{R}^{m\times p}$ y  $R\in\mathbb{R}^{p\times n}$ y con $Q$ una matriz ortogonal. 


Observe que $Q^TQ=I$ pero si $p<n$ entonces $QQ^T\not = I$. Ademas tenemos que $$\mbox{Span}(A)=\mbox{Span}\{ a_i\}_{i=1}^n=\mbox{Span}\{q_j\}_{j=1}^p=\mbox{Span}(Q).$$


Por otro lado, es común escribir la factorización **completa** $A=QR$ donde 
$$ A= [ Q_1 Q_2] \left[ \begin{array}{c}R_1\\ 0\end{array} \right]$$ 
de donde 
$$\mbox{Span}(A)=\mbox{Span}(Q_1)$$ y $$\mbox{Span}(A^T)^\perp=\mbox{Span}(Q_2).$$

Por últimos observe que si las columnas de $A$ son linealmente independientes, entonces $p=n$. 



# Gram - Schmidt Clásico

Considere el caso de columnas linealmente independientes, dados $\{a_1,a_2,\dots,a_n\}\subset \mathbb{R}^m$ linealmente independientes se desea generar $\{q_1,q_2,\dots,q_n\}$ tales que $q_i^Tq_j=\delta_{ij}$ y con las siguientes propiedades:

1. $\mbox{Span}\{a_1,a_2,\dots,a_n\} = \mbox{Span}\{\color{red}{q_1},a_2,\dots,a_n\}= \mbox{Span}\{\color{red}{q_1,q_2},\dots,a_n\}=\dots= = \mbox{Span}\{\color{red}{q_1,q_2,\dots,q_n}\} $.
2. $\mbox{Span}\{a_1,a_2,\dots,a_s\}=\mbox{Span}\{\color{red}{q_1,q_2,\dots,q_s}\}$ para $s=1,2,\dots,n$.

En el primer paso se tienen dos opciones para $q_1$, $q_1= \frac{a_1}{\|a_1\|}$ o $q_1= -\frac{a_1}{\|a_1\|}$.
Se fija una de ellas. Suponga que ya fueron contruidos $q_1,q_2,\dots,q_{i-1}$ tales que vale 1. y 2. arriba. Para calcular $q_i$ vemos que, si queremos que valga 2., debemos tener, 
$$
r_{ii}q_i=a_i-\sum_{\ell=1}^{i-1}q_\ell r_{\ell i}
$$
o 

$$
a_i= \sum_{\ell=1}^i q_\ell r_{\ell i}.
$$
Para calcular $r_{j,i}$ con $j=1,2,\dots,i-1,$ multiplicamos por $q_j^T$ para obtener ($q_j^Tq_\ell=\delta_{j,l}$)
$$
0=q_j^T(r_{ii}q_i)=q_j^T \Big(a_i-\sum_{\ell=1}^{i-1}q_\ell r_{\ell i} \Big) =q_j^Ta_i -r_{ji}
$$
de donde $r_{ji}= q_j^Ta_i$. Observe tambien que al multiplicar por $q_i^t$ obtenemos 
$$
r_{ii}=q_i^T(r_{ii}q_i)=q_i^T \Big(a_i-\sum_{\ell=1}^{i-1}q_\ell r_{\ell i} \Big) =q_j^Ta_i$$.

Al recopilar todas estas observaciones tenemos el algoritmo de la página 231 del texto guía.

In [1]:
function QRCGS(B)
    A=copy(B)
    sizeA=size(A)
    Q = zeros(sizeA) #(m,n)
    R = zeros(sizeA[2],sizeA[2]) #(n,n)
    for i = 1:sizeA[2]
        for j = 1:i-1
            R[j,i] = Q[:,j]'A[:,i]
        end
        p = A[:,i] - Q[:,1:i-1]*R[1:i-1,i]
        R[i,i]=norm(p)
#        if abs(R[i,i])<0.00000000001
#            println("Rii cercano 0")
#        end
        Q[:,i] = p/R[i,i]
    end
    return Q,R
end

QRCGS (generic function with 1 method)

Note que en la segunda línea del codigo copiamos la matriz $B$ a la matriz $A$. Esto no es necesario y se hace para una comparación mas justa con el algoritmo de MGS ya que el parametro de la función en Julia para por referencia (pass-by-sharing). Podemos verificar el algoritmo con los siguientes ejemplos.

In [10]:
n,m = 10,30
A1 = rand(m,n);

In [11]:
Q1, R1 = QRCGS(A1)
R1

10×10 Matrix{Float64}:
 3.35286  2.12533  2.50694  2.25671   …   2.08142    2.71324    2.11495
 0.0      1.76892  1.21566  1.07047       0.794583   0.594351   0.936165
 0.0      0.0      1.67555  0.609167      0.680001  -0.165425   0.965955
 0.0      0.0      0.0      1.72585       0.217051   0.24412    0.395337
 0.0      0.0      0.0      0.0           0.552704   0.23046    0.873462
 0.0      0.0      0.0      0.0       …   0.446252   0.604423   0.475433
 0.0      0.0      0.0      0.0          -0.501752   0.427444  -0.112004
 0.0      0.0      0.0      0.0           1.47162    0.555501   0.883773
 0.0      0.0      0.0      0.0           0.0        1.27097    0.0910166
 0.0      0.0      0.0      0.0           0.0        0.0        1.54215

In [12]:
Q1

30×10 Matrix{Float64}:
 0.131511    -0.0121406  -0.0774895    0.315244   …  -0.00951921   0.220507
 0.255914     0.0526289  -0.144903     0.0639997      0.0929361    0.234951
 0.29775     -0.243083    0.28262      0.214942      -0.0168047   -0.203435
 0.143324     0.353062    0.0159099    0.118712      -0.25865     -0.238388
 0.159866     0.153162   -0.0599877    0.0578231      0.157838    -0.157969
 0.0476022    0.128961    0.338587    -0.0388336  …   0.372583     0.272178
 0.262857    -0.0415933  -0.00792237  -0.0822349     -0.218222     0.0898094
 0.105394     0.203689    0.210265     0.0277987     -0.0236017    0.173814
 0.252327    -0.11741    -0.0313189   -0.0052933      0.111722     0.0178959
 0.21371      0.117026    0.143949    -0.21545       -0.133179    -0.215064
 0.0884334    0.0169797   0.300544    -0.14491    …  -0.145972     0.180472
 0.138785     0.299227    0.021213    -0.145506      -0.168011     0.241743
 0.254464    -0.129788    0.0954688   -0.171467       0.333098 

Para verificar la precisión de la factorización podemos calcular la norma del residuo $A-QR$  y para la ortogonaliad de los vectores podemos calcular la norma de $Q^TQ-I$. Para el ejemplo anterior tenemos los siguitnes resultados. 

In [13]:
opnorm(A1-Q1*R1)

2.0239453372712737e-16

In [14]:
opnorm(Q1'*Q1-UniformScaling(1))

3.3413250195755675e-15

En el siguiente ejemplo consideramos una matriz $A=[A_1,A_2]$ donde $A_2$ es una perturbación de $A_1$. 

In [73]:
n = 200
m = 1000 
A1 = rand(m,Int(n/2));
ϵ= 1E-5 # tamaño de la perturbación
A2=A1+ϵ*rand(m,Int(n/2));
A3=[A1 A2];

In [74]:
Q3, R3 = QRCGS(A3);

In [71]:
opnorm(A3-Q3*R3)

9.780219818316988e-16

In [75]:
opnorm(Q3'Q3-UniformScaling(1))

0.0002132279615192013

Aquí observamos la poca estabilidad numérica del CGS ya que el residuo de la ortonormalidad no es pequeño. 

# Gram - Schmidt modificado


Alternativamente al algoritmo anterior podemos proceder como sigue. Después de calcular $q_1=r_{11}a_1$, restamos la componente en $q_1$ de $a_2,a_3,\dots,n$, esto es hacemos 
$a_i=a_i-(q_1^Ta_i)q_1$. Obtenemos *nuevos* vectores $a_2,a_3,\dots,a_n$ ortogonales a $q_1$. Seguidamente calculamos $q_2=r_{22}a_2$ y repetimos el proceso anterior ahora con $a_3,\dots,a_n$. 

Considere el algoritmo de MGS. Algoritmos 5.2.5 de la página 231-232 del texto guía. 

In [78]:
function QRMGS(B)
    A=copy(B)
    sizeA=size(A)
    Q = zeros(sizeA) #(m,n)
    R = zeros(sizeA[2],sizeA[2]) #(n,n)
    for i = 1:sizeA[2]
        R[i,i] = norm(A[:,i])
        Q[:,i] = A[:,i]/R[i,i]
        for j = i + 1: sizeA[2]
            R[i,j] = Q[:,i]'A[:,j]
            A[:,j] = A[:,j] - Q[:,i]R[i,j]
        end
    end
    return Q, R
end

QRMGS (generic function with 1 method)

Recuerde que copiamos la matriz $B$ en $A$, lo que es necesario solo en el caso de que necesitemos retornar la matriz original $A$. Si no se necesita retornar $A$, entonces se configura el algoritmo para que rescriba la matriz $Q$ en la matriz $A$. 

Consideremos el siguiente ejemplo.

In [27]:
n = 30
m = 300 
A4 = rand(m,n);

In [28]:
Q4,R4=QRMGS(A4);

In [29]:
opnorm(Q4'*Q4-UniformScaling(1))

1.7919900260532545e-15

In [30]:
opnorm(A4-Q4*R4)

2.75039867065065e-15

# Comparación CGS vs MGS

In [31]:
using BenchmarkTools

In [32]:
n = 30
m = 300 
A = rand(m,n);

In [33]:
@benchmark Q,R=QRCGS(A)

BenchmarkTools.Trial: 3435 samples with 1 evaluation.
 Range (min … max):  776.762 μs … 12.049 ms  ┊ GC (min … max):  0.00% … 63.72%
 Time  (median):     954.704 μs              ┊ GC (median):     0.00%
 Time  (mean ± σ):     1.440 ms ±  1.150 ms  ┊ GC (mean ± σ):  13.75% ± 17.66%

  ▅█▅▄▄▂▁            ▂▂▁▂▂▂▁▁                                  ▁
  ████████▇▇▇▇▆▇▇▆▆█▇███████████▆▇▆▅▅▅▄▅▅▅▅▁▁▄▃▁▁▃▃▁▁▁▃▁▄▁▁▁▁▃ █
  777 μs        Histogram: log(frequency) by time      6.46 ms <

 Memory estimate: 3.56 MiB, allocs estimate: 1109.

In [34]:
@benchmark Q,R=QRMGS(A)

BenchmarkTools.Trial: 2173 samples with 1 evaluation.
 Range (min … max):  1.577 ms … 12.717 ms  ┊ GC (min … max):  0.00% … 53.27%
 Time  (median):     1.752 ms              ┊ GC (median):     0.00%
 Time  (mean ± σ):   2.285 ms ±  1.104 ms  ┊ GC (mean ± σ):  16.09% ± 20.10%

  ▄█▇▅▄▃▂▁▂▁                      ▃▄▃▁                       ▁
  ██████████▇▇▇▆▅▃▄▄▄▅▄▁▅▃▃▄▃▅▁▃▄▆██████▇▆█▇▇▇▇▇▆▅▆▃▆▅▅▄▅▁▁▄ █
  1.58 ms      Histogram: log(frequency) by time     5.65 ms <

 Memory estimate: 6.74 MiB, allocs estimate: 2706.

Consideremos ahora el caso de matrices con entradas cerca de ser linealmente dependientes. 

In [79]:
n = 200
m = 1000 
A5 = rand(m,Int(n/2));
ϵ=1E-5
A6=A5+ϵ*rand(m,Int(n/2));
#opnorm(A1-A2)
A7=[A5 A6];
#A3=rand(m,n)

In [80]:
Q7c,R7c=QRCGS(A7);

In [81]:
opnorm(Q7c'*Q7c-UniformScaling(1))

0.0002490809922457353

In [82]:
opnorm(A7-Q7c*R7c)

9.123698071186868e-16

In [83]:
Q7m,R7m=QRMGS(A7);

In [84]:
opnorm(Q7m'*Q7m-UniformScaling(1))

3.0498774317701587e-10

In [85]:
opnorm(A7-Q7m*R7m)

1.3733437393577418e-14

Observe que con el MGS los vectores estan más cerca de ser realmente orgotonales que con el CGS. Además, con el CGS el residuo de la factorización es un poco menor que con MGS. Finalmente hacemos la siguiente observación: cuando el residuo de ortogonalidad es grade podemos reortogonalizar. Suponga que inicamos con $A\approx QR$. Entonces podmeos calcular $Q\approx \tilde{Q}\tilde{R}$. Obtenemos $A\approx (\tilde{Q}\tilde{R})R=\tilde{Q}\hat{R}$. 

In [88]:
tildeQ7c,tildeR7c=QRCGS(Q7c);

In [90]:
opnorm(tildeQ7c'*tildeQ7c-UniformScaling(1))

7.953909381658018e-16

In [89]:
opnorm(A7 - tildeQ7c*(tildeR7c*R7c))

3.045277391483915e-14

Terminamos enunciado el siguiente resultado.

**Teorema:** Si las columnas de $A$ son linealmente independientes y $r_{ii}>0$ para $i=1,\dots,n$ entonces la factorización $A=Q_1R_1$ con $Q_1$ ortogonal y $R_1$ triangular superior es única. 

**Demostración:** Dado que $A=Q_1R_1$ implica $A^TA=(Q_1R_1)^T(Q_1R_1)= R_1^T ( Q_1^TQ_1)R_1=R_1^TR_1$, el resultado se sigue de la unicidad de la factorización de Cholesky para matrices positivas definidas, en este caso aplicada a $A^TA$. 